# Knowledge-based Architecture to Transactional User Sequence and Historical Affinity (KATUSHA)

Self-made hybrid architecture for learning user transactional patterns and historical affinities via embedding-based representation of tabular data

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
from tqdm import tqdm

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import sys
import os

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)
os.chdir(project_root)

from tecd_retail_recsys.data import DataPreprocessor
from tecd_retail_recsys.utils import get_avg_recs_price, get_item_to_price
from tecd_retail_recsys.metrics import calculate_metrics
from tecd_retail_recsys.models import TopPopular, TopPersonal

from tecd_retail_recsys.models.katusha import (
    transform_frame,
    RecDataset,
    TwoLayerRecMLP,
    train_one_epoch,
    evaluate,
    predict_scores,
    transform_frame_inference,
    predict_scores_inference
)

from torch.utils.data import DataLoader
import torch
import torch.nn as nn

from copy import deepcopy

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

INFO:faiss.loader:Loading faiss.
INFO:faiss.loader:Successfully loaded faiss.


In [3]:
dp = DataPreprocessor(day_begin=1082, day_end=1308, val_days=20, test_days=20, min_user_interactions=1, min_item_interactions=20)
train_df, val_df, test_df = dp.preprocess()
joined = dp.get_grouped_data(train_df, val_df, test_df)
joined['train_val_interactions'] = joined['train_interactions'] + joined['val_interactions']
print(joined.shape)

Starting data preprocessing...
Loading events from t_ecd_small_partial/dataset/small/retail/events
Loaded 267,043,972 total events
Loading items data from t_ecd_small_partial/dataset/small/retail/items.pq
Loaded 250,171 items with features: ['item_id', 'item_brand_id', 'item_category', 'item_subcategory', 'item_price', 'item_embedding']
Merged item features. Data shape: (267043972, 12)
Filtered to 3,852,145 events with action_type='added_to_cart'
After filtering (min_user_interactions=1, min_item_interactions=20): 3,327,057 events, 84,830 users, 32,095 items
Created mappings: 84830 users, 32095 items
Temporal split - Train: days < 1269 (922,368 events), Val: days 1269-1288 (232,900 events), Test: days >= 1289 (227,959 events)
Users in each part (train, val, test) - 7422
(7422, 5)


In [4]:
def fill_to_k(df, primary_col, backup_col, target_len=100, out_col=None):
    out_col = out_col or primary_col

    def merge_lists(row):
        a = list(row[primary_col]) if row[primary_col] is not None else []
        b = list(row[backup_col]) if row[backup_col] is not None else []

        if len(a) >= target_len:
            return a[:target_len]

        need = target_len - len(a)
        return a + b[:need]

    df[out_col] = df.apply(merge_lists, axis=1)
    return df


In [37]:
TRAIN_DF, VAL_DF = train_df[train_df['day'] < 1268 - 19], train_df[(train_df['day'] >= 1268 - 19)] 
JOINED = dp.get_grouped_data(TRAIN_DF, VAL_DF, VAL_DF)

In [38]:
toppers_val = TopPersonal()
toppers_val.fit(JOINED, col='train_interactions')
JOINED['toppers_recs'] = toppers_val.predict(JOINED, topn=100)

toppop_val = TopPopular()
toppop_val.fit(JOINED, col='train_interactions')
JOINED['toppopular_recs_val'] = toppop_val.predict(JOINED, topn=100, return_scores=False)

JOINED = fill_to_k(
    JOINED,
    primary_col='toppers_recs',
    backup_col='toppopular_recs_val',
    target_len=100,
    out_col='toppers_recs_filled'
)

In [39]:
features_df = JOINED[['user_id', 'toppers_recs_filled']].explode('toppers_recs_filled')
features_df.rename(columns={'toppers_recs_filled': 'item_id'}, inplace=True)

VAL_DF['target'] = 1
features_df = features_df.merge(VAL_DF[['user_id', 'item_id', 'target']].drop_duplicates(), how='left').fillna(0)

from tecd_retail_recsys.data.features_builder import CandidateFeatureBuilder
builder = CandidateFeatureBuilder(TRAIN_DF)
features_df = builder.fit_transform(
    features_df,
    feature_groups=("user", "user_item", "item")
)

for col in ['item_brand_id', 'item_category', 'item_subcategory']:
    features_df[col] = features_df[col].cat.add_categories(['unknown']).fillna('unknown')

categorical_features = [
    "user_id",
    "item_id",
    "item_brand_id",
    "item_category",
    "item_subcategory"
]

features_df['user_id'] = features_df['user_id'].astype(np.int64)
features_df['item_id'] = features_df['item_id'].astype(np.int64)
features_df['item_brand_id'] = features_df['item_brand_id'].astype(str)

In [40]:
TARGET_COL = "target"

categorical_features = [
    "user_id",
    "item_id",
    "item_brand_id",
    "item_category",
    "item_subcategory",
]

features_df["user_id"] = features_df["user_id"].astype(np.int64)
features_df["item_id"] = features_df["item_id"].astype(np.int64)
features_df["item_brand_id"] = features_df["item_brand_id"].astype(str)
features_df["item_category"] = features_df["item_category"].astype(str)
features_df["item_subcategory"] = features_df["item_subcategory"].astype(str)

all_feature_cols = [c for c in features_df.columns if c != TARGET_COL]
numeric_features = [c for c in all_feature_cols if c not in categorical_features]

print("Categorical:", categorical_features)
print("Numeric:", numeric_features)

Categorical: ['user_id', 'item_id', 'item_brand_id', 'item_category', 'item_subcategory']
Numeric: ['user_events_total', 'user_unique_items', 'user_unique_brands', 'user_unique_categories', 'user_unique_subcategories', 'user_first_day', 'user_last_day', 'user_first_ts', 'user_last_ts', 'user_avg_price', 'user_median_price', 'user_min_price', 'user_max_price', 'user_price_std', 'user_active_days', 'user_events_per_active_day', 'user_repeat_item_ratio', 'user_item_interactions', 'user_item_first_day', 'user_item_last_day', 'user_item_first_ts', 'user_item_last_ts', 'user_item_active_span_days', 'user_item_repeat_flag', 'top_personal_score', 'item_events_total', 'item_unique_users', 'item_first_day', 'item_last_day', 'item_first_ts', 'item_last_ts', 'item_avg_price', 'item_median_price', 'item_min_price', 'item_max_price', 'item_price_std', 'item_active_days', 'item_events_per_active_day', 'item_repeat_user_ratio', 'item_last_price', 'item_price_vs_user_avg', 'item_price_z_vs_user', 'user

In [41]:
unique_users = features_df["user_id"].drop_duplicates().values
rng = np.random.default_rng(42)
rng.shuffle(unique_users)

val_frac = 0.15
val_size = int(len(unique_users) * val_frac)
val_users = set(unique_users[:val_size])

train_mask = ~features_df["user_id"].isin(val_users)
val_mask = features_df["user_id"].isin(val_users)

train_df_nn = features_df.loc[train_mask].reset_index(drop=True).copy()
val_df_nn = features_df.loc[val_mask].reset_index(drop=True).copy()

print(train_df_nn.shape, val_df_nn.shape)
print(train_df_nn[TARGET_COL].mean(), val_df_nn[TARGET_COL].mean())

(409200, 53) (72200, 53)
0.07162267839687195 0.07034626038781164


In [42]:
cat_maps = {}
cat_cardinalities = {}

for col in categorical_features:
    vals = train_df_nn[col].astype(str).fillna("__NA__").unique().tolist()
    mapping = {v: i + 1 for i, v in enumerate(vals)}
    cat_maps[col] = mapping
    cat_cardinalities[col] = len(mapping) + 1

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
scaler.fit(train_df_nn[numeric_features].fillna(0.0).astype(np.float32))

,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True


In [43]:
X_cat_train, X_num_train, y_train = transform_frame(
    train_df_nn, categorical_features, numeric_features, cat_maps, scaler
)
X_cat_val, X_num_val, y_val = transform_frame(
    val_df_nn, categorical_features, numeric_features, cat_maps, scaler
)

print(X_cat_train.shape, X_num_train.shape, y_train.shape)
print(X_cat_val.shape, X_num_val.shape, y_val.shape)

(409200, 5) (409200, 47) (409200,)
(72200, 5) (72200, 47) (72200,)


In [44]:
train_ds = RecDataset(X_cat_train, X_num_train, y_train)
val_ds = RecDataset(X_cat_val, X_num_val, y_val)

train_loader = DataLoader(train_ds, batch_size=4096, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=8192, shuffle=False, num_workers=0)

In [61]:
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.mps.is_available() else "cpu")
print(device)

model = TwoLayerRecMLP(
    cat_cardinalities=cat_cardinalities,
    num_numeric_features=len(numeric_features),
    hidden1=256,
    hidden2=128,
    dropout=0.15
).to(device)

pos = y_train.sum()
neg = len(y_train) - pos
pos_weight = torch.tensor([neg / max(pos, 1.0)], dtype=torch.float32, device=device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)

mps


In [62]:
best_state = None
best_val_loss = float("inf")
patience = 3
patience_left = patience
n_epochs = 15

for epoch in range(1, n_epochs + 1):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_auc = evaluate(model, val_loader, criterion, device)

    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={train_loss:.5f} | "
        f"val_loss={val_loss:.5f} | "
        f"val_auc={val_auc:.5f}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = deepcopy(model.state_dict())
        patience_left = patience
    else:
        patience_left -= 1
        if patience_left == 0:
            print("Early stopping")
            break

model.load_state_dict(best_state)

Epoch 01 | train_loss=1.00390 | val_loss=0.97259 | val_auc=0.81823
Epoch 02 | train_loss=0.92697 | val_loss=0.96999 | val_auc=0.82093
Epoch 03 | train_loss=0.88111 | val_loss=0.99334 | val_auc=0.81765
Epoch 04 | train_loss=0.83359 | val_loss=0.98525 | val_auc=0.81537
Epoch 05 | train_loss=0.78615 | val_loss=1.02244 | val_auc=0.80946
Early stopping


<All keys matched successfully>

In [63]:
X_scored = features_df.copy()
X_scored["score"] = predict_scores(
    model=model,
    df=X_scored,
    categorical_features=categorical_features,
    numeric_features=numeric_features,
    cat_maps=cat_maps,
    scaler=scaler,
    device=device,
)

val_recs = (
    X_scored.sort_values(by=["user_id", "score"], ascending=[True, False])[["user_id", "item_id"]]
    .groupby("user_id", as_index=False)
    .agg(list)
)

val_recs.rename(columns={"item_id": "katusha_val_recs"}, inplace=True)

In [64]:
katusha_val_metrics = calculate_metrics(
    joined.merge(val_recs),
    model_preds="katusha_val_recs",
    gt_col="val_interactions",
    train_col="train_interactions",
    k_values=[100],
    verbose=True
)

[Metrics debug] resolved gt_col='val_interactions' item_id_index=0
[Metrics debug] ratings_true shape: (169501, 3) ratings_pred shape: (481400, 3)
  ratings_true dtypes: {'user_id': dtype('int64'), 'item_id': dtype('int64')}
  ratings_pred dtypes: {'user_id': dtype('int64'), 'item_id': dtype('int64')}
  user_id=30 gt_count=9 pred_count=100 overlap=2
  user_id=39 gt_count=8 pred_count=96 overlap=2
  user_id=51 gt_count=24 pred_count=99 overlap=6

At k=100:
  MAP@100       = 0.1514
  NDCG@100      = 0.4236
  Precision@100 = 0.0844
  Recall@100    = 0.2220

Other Metrics:
  MRR                 = 0.4276
  Catalog Coverage    = 0.9586
  Diversity     = 0.9967  [0=same recs for all, 1=unique recs]
  Novelty             = 0.6603
  Serendipity         = 0.0161


### Обучим на трейн+вал

In [65]:
toppers_test = TopPersonal()
toppers_test.fit(joined, col='train_val_interactions')
joined['toppers_recs'] = toppers_test.predict(joined, topn=100)

# фолбэк
toppop_test = TopPopular()
toppop_test.fit(joined, col='train_val_interactions')
joined['toppopular_recs'] = toppop_test.predict(joined, topn=100, return_scores=False)

joined = fill_to_k(
    joined,
    primary_col='toppers_recs',
    backup_col='toppopular_recs',
    target_len=100,
    out_col='toppers_recs_filled'
)

In [66]:
features_df = joined[['user_id', 'toppers_recs_filled']].explode('toppers_recs_filled')
features_df.rename(columns={'toppers_recs_filled': 'item_id'}, inplace=True)

builder = CandidateFeatureBuilder(pd.concat([train_df, val_df]))
features_df = builder.fit_transform(
    features_df,
    feature_groups=("user", "user_item", "item")
)

for col in ['item_brand_id', 'item_category', 'item_subcategory']:
    features_df[col] = features_df[col].cat.add_categories(['unknown']).fillna('unknown')

categorical_features = [
    "user_id",
    "item_id",
    "item_brand_id",
    "item_category",
    "item_subcategory"
]

features_df['user_id'] = features_df['user_id'].astype(np.int64)
features_df['item_id'] = features_df['item_id'].astype(np.int64)
features_df['item_brand_id'] = features_df['item_brand_id'].astype(str)

In [67]:
train_val_scored = features_df.copy()

train_val_scored["score"] = predict_scores_inference(
    model=model,
    df=train_val_scored,
    categorical_features=categorical_features,
    numeric_features=numeric_features,
    cat_maps=cat_maps,
    scaler=scaler,
    device=device,
    batch_size=8192
)

train_val_scored.head()

,user_id,item_id,user_events_total,user_unique_items,user_unique_brands,user_unique_categories,user_unique_subcategories,user_first_day,user_last_day,user_first_ts,...,item_subcategory,item_last_price,item_price_vs_user_avg,item_price_z_vs_user,user_item_share,item_popularity_per_user,is_cold_user,is_cold_item,is_new_item_for_user,score
0,12,19043,88,74,5,4,19,1104,1282,95398378,...,Dairy Products and Eggs,-4.551,-0.163364,-0.152734,0.034091,1.333333,0,0,0,0.945856
1,12,15381,88,74,5,4,19,1104,1282,95398378,...,Food Products,-4.944,-0.556364,-0.520160,0.034091,1.136364,0,0,0,0.969845
2,12,17157,88,74,5,4,19,1104,1282,95398378,...,Dairy Products and Eggs,-4.750,-0.362364,-0.338784,0.022727,1.846154,0,0,0,0.966370
3,12,2204,88,74,5,4,19,1104,1282,95398378,...,"Grains, Pasta, and Flour Products",-4.982,-0.594364,-0.555688,0.022727,1.253846,0,0,0,0.943923
4,12,784,88,74,5,4,19,1104,1282,95398378,...,Food Products,-4.946,-0.558364,-0.522030,0.022727,1.133333,0,0,0,0.960600


In [70]:
TOP_K = 100

train_val_recs = (
    train_val_scored
    .sort_values(["user_id", "score"], ascending=[True, False])
    .groupby("user_id", as_index=False)
    .agg({"item_id": list})
)

train_val_recs["katusha_test_recs"] = train_val_recs["item_id"].apply(lambda x: x[:TOP_K])
train_val_recs = train_val_recs[["user_id", "katusha_test_recs"]]

In [71]:
katusha_test_metrics = calculate_metrics(
    joined.merge(train_val_recs),
    model_preds="katusha_test_recs",
    gt_col="test_interactions",
    train_col="train_val_interactions",
    k_values=[100],
    verbose=True
)

[Metrics debug] resolved gt_col='test_interactions' item_id_index=0
[Metrics debug] ratings_true shape: (227959, 3) ratings_pred shape: (742200, 3)
  ratings_true dtypes: {'user_id': dtype('int64'), 'item_id': dtype('int64')}
  ratings_pred dtypes: {'user_id': dtype('int64'), 'item_id': dtype('int64')}
  user_id=12 gt_count=10 pred_count=94 overlap=3
  user_id=15 gt_count=54 pred_count=97 overlap=2
  user_id=23 gt_count=38 pred_count=98 overlap=12

At k=100:
  MAP@100       = 0.1616
  NDCG@100      = 0.4255
  Precision@100 = 0.0851
  Recall@100    = 0.2531

Other Metrics:
  MRR                 = 0.4301
  Catalog Coverage    = 0.9954
  Diversity     = 0.9969  [0=same recs for all, 1=unique recs]
  Novelty             = 0.6567
  Serendipity         = 0.0136


In [ ]:
item_to_price = get_item_to_price(dp)
joined = joined.merge(train_val_recs)
avg_recs_price = get_avg_recs_price(joined, item_to_price, 'katusha_test_recs')
print(f'Средняя цена рекомендаций модели KATUSHA: {avg_recs_price:.2f}')

Средняя цена рекомендаций модели KATUSHA: -3.98
